# Test: Simplexity HMM generation

This notebook tests the [simplexity](https://github.com/Astera-org/simplexity) library for HMM-based sequence generation, aligned with the geometric interpretability project (belief state geometry, Mess3, etc.).

**Simplexity** provides:
- `build_hidden_markov_model(process_name, process_params)` — build HMMs (e.g. `"mess3"`, `"coin"`, `"even_ones"`, `"rrxor"`, …)
- `generate_data_batch` — generate sequences (and optionally belief-state history) from an HMM
- Belief-state updates and observation probability distributions

Install from GitHub if needed:
```bash
pip install "simplexity @ git+https://github.com/Astera-org/simplexity.git"
```
(Requires Python ≥3.12; pulls in JAX, Equinox, etc.)

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

## 1. Build an HMM (Mess3)

Mess3 is a 3-state, 3-symbol HMM used in computational mechanics. Parameters:
- `x`: controls transition structure
- `a`: asymmetry (in simplexity the param is `a`; in our `Mess3HMM` it was `alpha`)

We use `x=0.05`, `a=0.85` to stay close to the project's existing Mess3 setup.

In [ ]:
from simplexity.generative_processes.builder import build_hidden_markov_model

hmm = build_hidden_markov_model(
    "mess3",
    process_params={"x": 0.05, "a": 0.85},
    device=None,
)

print(f"Vocab size (symbols): {hmm.vocab_size}")
print(f"Num hidden states: {hmm.num_states}")
print(f"Initial state (belief): {np.array(hmm.initial_state)}")

## 2. Generate sequences

Use `generate_data_batch` to sample sequences. It returns `(gen_states, inputs, labels)` where `inputs`/`labels` are consecutive tokens (for next-token prediction).

In [ ]:
from simplexity.generative_processes.generator import generate_data_batch

batch_size = 4
sequence_len = 50
key = jax.random.PRNGKey(42)

# Initial states: one copy of HMM initial state per batch item
gen_states = jnp.tile(hmm.initial_state, (batch_size, 1))

final_states, inputs, labels = generate_data_batch(
    gen_states, hmm, batch_size, sequence_len, key
)

print("inputs shape:", inputs.shape)   # (batch_size, sequence_len)
print("labels shape:", labels.shape)   # (batch_size, sequence_len)
print("Sample sequence (first batch, first 20 tokens):", np.array(inputs[0, :20]))
print("Token map: 0=A, 1=B, 2=C for Mess3")

## 3. Generate with full belief-state history

Use `generate_data_batch_with_full_history` to get per-token belief states (for probing / visualization).

In [ ]:
from simplexity.generative_processes.generator import generate_data_batch_with_full_history

key2 = jax.random.PRNGKey(123)
gen_states2 = jnp.tile(hmm.initial_state, (batch_size, 1))

next_states, belief_states, prefix_probs, inputs_h, labels_h = generate_data_batch_with_full_history(
    gen_states2, hmm, batch_size, sequence_len, key2
)

print("belief_states shape:", belief_states.shape)  # (batch_size, sequence_len, num_states)
print("Belief state at t=0 (first batch):", np.array(belief_states[0, 0]))
print("Belief state at t=10 (first batch):", np.array(belief_states[0, 10]))

## 4. Observation distribution and sequence probability

Compute P(observation | belief state) and P(sequence) under the HMM.

In [ ]:
# Observation distribution from initial state
obs_probs = hmm.observation_probability_distribution(hmm.initial_state)
print("P(symbol | initial state):", np.array(obs_probs))

# Log-probability of one generated sequence
seq = inputs[0]
log_prob = hmm.log_probability(seq)
print(f"Log P(sequence): {float(log_prob):.4f}")

## 5. Other HMMs: coin, even_ones

Simplexity includes several built-in processes. Quick sanity check.

In [ ]:
# Coin process: P(head)=p, vocab size 2
coin_hmm = build_hidden_markov_model("coin", process_params={"p": 0.7})
print("Coin HMM vocab_size:", coin_hmm.vocab_size, "num_states:", coin_hmm.num_states)

gs = jnp.tile(coin_hmm.initial_state, (2, 1))
_, inp, _ = generate_data_batch(gs, coin_hmm, 2, 20, jax.random.PRNGKey(0))
print("Coin sample sequence:", np.array(inp[0]))

# Even ones: 2 states, 2 symbols
eo_hmm = build_hidden_markov_model("even_ones", process_params={"p": 0.5})
print("\nEven-ones HMM vocab_size:", eo_hmm.vocab_size, "num_states:", eo_hmm.num_states)

## 6. Map simplexity tokens to project symbols (A, B, C)

For use with the rest of the repo (e.g. prompting an LLM), map integer tokens to the same symbols as `Mess3HMM`.

In [ ]:
HMM_CONCEPTS = ['A', 'B', 'C']  # same as accuracy_vs_context_HMM.ipynb

def tokens_to_symbols(tokens: jnp.ndarray) -> list:
    """Convert integer token array to list of symbols (e.g. for Mess3)."""
    return [HMM_CONCEPTS[int(t)] for t in tokens]

sample_tokens = np.array(inputs[0, :15])
print("First 15 tokens as symbols:", tokens_to_symbols(sample_tokens))